# Ricker 子波示意图

对应 `note/深度域物理约束神经网络.md` 中第 2 页和第 13 页的“假设一个非常简单的子波”。

运行下面两个代码单元会直接在 notebook 中显示图片，不会把图片保存到本地文件。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def ricker(t_ms, f_hz):
    """Standard zero-phase Ricker wavelet, normalized to w(0)=1."""
    t_s = t_ms / 1000.0
    a = (np.pi * f_hz * t_s) ** 2
    return (1.0 - 2.0 * a) * np.exp(-a)


def solve_ricker_frequency(offset_ms, target_amp):
    """Find the Ricker frequency whose positive side sample reaches target_amp."""
    lo, hi = 0.0, 0.5
    for _ in range(80):
        mid = (lo + hi) / 2.0
        amp = (1.0 - 2.0 * mid) * np.exp(-mid)
        if amp > target_amp:
            lo = mid
        else:
            hi = mid
    a = (lo + hi) / 2.0
    return np.sqrt(a) / (np.pi * offset_ms / 1000.0)


def generalized_ricker_like(t_ms, b, c, sample_ms=10.0):
    """Even, Ricker-shaped curve used to pass through the slide's five teaching samples."""
    x = t_ms / sample_ms
    return (1.0 - c * x**2) * np.exp(-b * x**2)


def solve_generalized_params(y1=0.2, y2=-0.1):
    """Solve b,c so y(1)=y1 and y(2)=y2 for the generalized Ricker-like curve."""

    def value_at_two_minus_target(b):
        c = 1.0 - y1 * np.exp(b)
        return (1.0 - 4.0 * c) * np.exp(-4.0 * b) - y2

    lo, hi = 0.0, 2.0
    for _ in range(80):
        mid = (lo + hi) / 2.0
        if value_at_two_minus_target(mid) < 0.0:
            lo = mid
        else:
            hi = mid
    b = (lo + hi) / 2.0
    c = 1.0 - y1 * np.exp(b)
    return b, c


def style_axis(ax, title):
    ax.axhline(0.0, color="#2f3542", linewidth=1.0)
    ax.axvline(0.0, color="#747d8c", linewidth=1.0, linestyle="--", alpha=0.75)
    ax.grid(True, color="#dfe4ea", linewidth=0.8, alpha=0.9)
    ax.set_title(title, fontsize=16, pad=12)
    ax.set_xlabel("时间差 / ms", fontsize=12)
    ax.set_ylabel("振幅", fontsize=12)
    ax.set_ylim(-0.52, 1.18)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


def mark_samples(ax, x, y, label_offset=0.075):
    ax.scatter(x, y, s=82, color="#e74c3c", edgecolor="white", linewidth=1.2, zorder=4)
    for xi, yi in zip(x, y):
        va = "bottom" if yi >= 0 else "top"
        dy = label_offset if yi >= 0 else -label_offset
        ax.annotate(
            f"{yi:.1f}",
            xy=(xi, yi),
            xytext=(xi, yi + dy),
            ha="center",
            va=va,
            fontsize=12,
            fontweight="bold",
            color="#c0392b",
        )

## 第 2 页：三点版本

标注点为 `w(-10)=0.2, w(0)=1.0, w(+10)=0.2`。


In [ ]:
lags_page2 = np.array([-10.0, 0.0, 10.0])
amps_page2 = np.array([0.2, 1.0, 0.2])

f_page2 = solve_ricker_frequency(offset_ms=10.0, target_amp=0.2)
t_page2 = np.linspace(-42.0, 42.0, 900)
w_page2 = ricker(t_page2, f_page2)

fig, ax = plt.subplots(figsize=(7.2, 5), dpi=160)
ax.plot(t_page2, w_page2, color="#1f6f8b", linewidth=2.8)
ax.vlines(lags_page2, 0.0, amps_page2, color="#e74c3c", linewidth=1.4, alpha=0.7)
mark_samples(ax, lags_page2, amps_page2)
style_axis(ax, "三采样点的 Ricker 子波示意图")
ax.set_xticks([-40, -30, -20, -10, 0, 10, 20, 30, 40])
plt.show()

## 第 13 页：五点版本

标注点为 `w(-20)=-0.1, w(-10)=0.2, w(0)=1.0, w(+10)=0.2, w(+20)=-0.1`。


In [ ]:
lags_page13 = np.array([-20.0, -10.0, 0.0, 10.0, 20.0])
amps_page13 = np.array([-0.1, 0.2, 1.0, 0.2, -0.1])

b_page13, c_page13 = solve_generalized_params(y1=0.2, y2=-0.1)
t_page13 = np.linspace(-34.0, 34.0, 900)
w_page13 = generalized_ricker_like(t_page13, b_page13, c_page13)

fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=160)
ax.plot(t_page13, w_page13, color="#1f6f8b", linewidth=2.8)
ax.vlines(lags_page13, 0.0, amps_page13, color="#e74c3c", linewidth=1.4, alpha=0.7)
mark_samples(ax, lags_page13, amps_page13)
style_axis(ax, "五采样点的 Ricker 子波示意图")
ax.set_xticks([-30, -20, -10, 0, 10, 20, 30])
plt.show()